In [ ]:
import pyguidos
from pyguidos import data

import rasterio
import numpy as np
import os
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.image as mpimg
from matplotlib.colors import ListedColormap, BoundaryNorm

In [ ]:
data_dir = data.test_data_dir()

in_dir = data_dir / 'binarymap'
out_dir = Path('/home/user/tmp/output/') # <<< REPLACE with your actual desired output path
                                         #     The folder must be empty

In [ ]:
bin_map = input_dir / 'binarymap.tif'

with rasterio.open(bin_map) as src:
    array = src.read(1)

fig, ax = plt.subplots(figsize=(10, 8))

colors = ['white', 'lightgrey', 'darkgreen'] 
cmap = ListedColormap(colors)
im = ax.imshow(array, cmap=cmap, vmin=-0.5, vmax=2.5)
legend_labels = {
    0: 'Missing/NoData', 
    1: 'Background',     
    2: 'Foreground'      
}

patches = []
for val in sorted(legend_labels.keys()): 
    label = legend_labels[val]
    color = colors[val] 
    patch = mpatches.Patch(color=color, label=label)
    patches.append(patch)
legend = ax.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

plt.axis('off')
plt.show()

In [ ]:

pyguidos.gwb_mspa(in_dir, out_dir, conn_8=True, wdge_width=1, transition=True, int_ext=True, stats=True)

In [ ]:
map_name = str(bin_map).split('/')[-1][:-4]
res_dir = os.path.join(out_dir, map_name + '_mspa')

mspa_map = os.path.join(res_dir, map_name + '_8_1_1_1.tif')
mspa_map_txt = os.path.join(res_dir, map_name + '_8_1_1_1.txt')

In [ ]:
with rasterio.open(mspa_map) as src:
    array = src.read(1)

unique_values = np.unique(array).tolist()

class_scheme = [
    {'name': 'Core', 'values': [17, 117], 'color': '#00C800', 'label': 'Core (17/117)'}, 
    {'name': 'Islet', 'values': [9, 109], 'color': '#A03C00', 'label': 'Islet (9/109)'}, 
    {'name': 'Perforation', 'values': [5, 105], 'color': '#0000FF', 'label': 'Perforation (5/105)'}, 
    {'name': 'Edge', 'values': [3, 103], 'color': '#000000', 'label': 'Edge (3/103)'}, 
    {'name': 'Loop', 'values': [65, 165], 'color': '#FFFF00', 'label': 'Loop (65/165)'}, 
    {'name': 'Loop in Edge', 'values': [67, 167], 'color': '#FFFF00', 'label': 'Loop in Edge (67/167)'}, 
    {'name': 'Loop in Perforation', 'values': [69, 169], 'color': '#FFFF00', 'label': 'Loop in Perforation (69/169)'}, 
    {'name': 'Bridge', 'values': [33, 133], 'color': '#FF0000', 'label': 'Bridge (33/133)'},
    {'name': 'Bridge in Edge', 'values': [35, 135], 'color': '#FF0000', 'label': 'Bridge in Edge (35/135)'}, 
    {'name': 'Bridge in Perforation', 'values': [37, 137], 'color': '#FF0000', 'label': 'Bridge in Perforation (37/137)'},
    {'name': 'Branch', 'values': [1, 101], 'color': '#FF8C00', 'label': 'Branch (1/101)'}, 
    {'name': 'Background', 'values': [0], 'color': '#DCDCDC', 'label': 'Background (0)'}, 
    {'name': 'Border-Opening', 'values': [220], 'color': '#C2C2C2', 'label': 'Border-Opening (220)'}, 
    {'name': 'Core-Opening', 'values': [100], 'color': '#888888', 'label': 'Core-Opening (100)'}, 
    {'name': 'No Data', 'values': [129], 'color': '#FFFFFF', 'label': 'No Data (129)'} 
]

byte_values = sorted(list(set(val for entry in class_scheme for val in entry['values'])))
value_to_color_map_for_scheme = {} 
for category in class_scheme:
    for val in category['values']:
        value_to_color_map_for_scheme[val] = category['color']

norm_boundaries = []
cmap_colors = []
norm_boundaries.append(byte_values[0] - 0.5)
for val in byte_values:
    cmap_colors.append(value_to_color_map_for_scheme.get(val, '#808080'))
    norm_boundaries.append(val + 0.5)
cmap = ListedColormap(cmap_colors)
norm = BoundaryNorm(norm_boundaries, cmap.N)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(array, cmap=cmap, norm=norm, interpolation='nearest')

categories = set()
for val in unique_values:
    found_category = False
    for category in class_scheme:
        if val in category['values']:
            categories.add(category['name'])
            found_category = True
            break

patches = []
for category in class_scheme:
    if category['name'] in categories:
        patch = mpatches.Patch(color=category['color'], label=category['label'])
        patches.append(patch)

legend = ax.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

plt.axis('off')
plt.show()

In [ ]:
with open(mspa_map_txt, 'r', encoding='utf-8') as f:
    txt = f.read()
    print(txt)